# Intent and slot baseline
Portfolio edition, 31 October 2026. Original coursework is preserved outside this folder.
Run from this notebook folder after installing requirements and placing the documented inputs in data/.
This baseline/audit is separate from the original coursework scores. All performance labels refer to its own holdout.


In [1]:
"""Small reproducible baseline, not a rerun of the saved BERT/BiLSTM experiments."""
from pathlib import Path
import json,re
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score,f1_score
import sklearn_crfsuite
from seqeval.metrics import f1_score as entity_f1
ROOT=Path.cwd() if (Path.cwd()/'data/nlp').exists() else Path(__file__).resolve().parent
rows=[]
for line in (ROOT/'data/nlp/train.txt').read_text().splitlines():
    if not line.strip():continue
    text,intent=line.rsplit('<=>',1);pairs=[x.rsplit(':',1) for x in text.split()];words=[p[0] for p in pairs];slots=[p[1] for p in pairs]
    assert len(words)==len(slots)
    rows.append({'words':words,'slots':slots,'intent':intent.strip(),'group':' '.join(w.lower() for w in words)})
groups=[r['group'] for r in rows];tr,va=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(rows,groups=groups));train=[rows[i] for i in tr];valid=[rows[i] for i in va]
assert not ({r['group'] for r in train}&{r['group'] for r in valid})
model=make_pipeline(TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True),LogisticRegression(max_iter=600,C=4,random_state=42))
model.fit([' '.join(r['words']) for r in train],[r['intent'] for r in train]);yp=model.predict([' '.join(r['words']) for r in valid]);yt=[r['intent'] for r in valid]
def token_features(words,i):
    w=words[i];f={'bias':1.0,'word':w.lower(),'suffix2':w[-2:],'suffix3':w[-3:],'digit':w.isdigit(),'upper':w.isupper(),'first':i==0,'last':i==len(words)-1}
    if i:f['prev']=words[i-1].lower()
    if i+1<len(words):f['next']=words[i+1].lower()
    return f
def features(r):return[token_features(r['words'],i) for i in range(len(r['words']))]
crf=sklearn_crfsuite.CRF(algorithm='lbfgs',c1=.1,c2=.1,max_iterations=80,all_possible_transitions=True)
crf.fit([features(r) for r in train],[r['slots'] for r in train]);sp=crf.predict([features(r) for r in valid]);st=[r['slots'] for r in valid]
flat_true=[x for seq in st for x in seq];flat_pred=[x for seq in sp for x in seq]
per=[]
for intent in sorted(set(yt)):
    idx=[i for i,v in enumerate(yt) if v==intent];correct=sum(yp[i]==intent for i in idx)
    per.append({'intent':intent,'support':len(idx),'recall':correct/len(idx)})
out={'kind':'new TF-IDF logistic intent classifier + CRF slot baseline','split':'80/20 grouped holdout; exact normalised utterances kept together; seed 42','train_examples':len(train),'validation_examples':len(valid),'group_overlap':0,'intent_accuracy':float(accuracy_score(yt,yp)),'intent_macro_f1':float(f1_score(yt,yp,average='macro',zero_division=0)),'slot_weighted_token_f1_including_O':float(f1_score(flat_true,flat_pred,average='weighted',zero_division=0)),'entity_f1':float(entity_f1(st,sp,zero_division=0)),'intent_details':sorted(per,key=lambda r:r['support'],reverse=True),'examples':[{'text':' '.join(valid[i]['words']),'true_intent':yt[i],'predicted_intent':str(yp[i]),'slots':[{'word':w,'truth':a,'prediction':b} for w,a,b in zip(valid[i]['words'],st[i],sp[i])]} for i in range(min(8,len(valid)))]}
(ROOT/'outputs').mkdir(exist_ok=True);(ROOT/'outputs/nlp-results.json').write_text(json.dumps(out,indent=2));print(json.dumps({k:v for k,v in out.items() if k not in ['intent_details','examples']},indent=2))


{
  "kind": "new TF-IDF logistic intent classifier + CRF slot baseline",
  "split": "80/20 grouped holdout; exact normalised utterances kept together; seed 42",
  "train_examples": 3586,
  "validation_examples": 892,
  "group_overlap": 0,
  "intent_accuracy": 0.9316143497757847,
  "intent_macro_f1": 0.5658160749026032,
  "slot_weighted_token_f1_including_O": 0.9686483781530446,
  "entity_f1": 0.9276425475158386
}
